In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

In [2]:
PESO_SATELITE_REBOTE = 0.65   # fracción del capital para Satélite + Rebote (60% a 70%)
VOL_MAX_NUCLEO = 0.35
VOL_MAX_SATELITE = 0.70       # bajar a 0.35 si quieres el filtro original
RSI_MAX_NUCLEO = 70
RSI_MAX_SATELITE = None       # None = sin límite de RSI (perseguir momentum)
MIS_POSICIONES = ["AAPL","AMD","GMEXICOB","NVDA","PE&OLES"]   
MIS_REBOTE = ["NVDA","PE&OLES"] 

In [3]:
universo = {
    "AMX": "AMX", "GMEXICO": "GMEXICOB.MX", "PE&OLES": "PE&OLES.MX",
    "CRM": "CRM.MX", "META": "META.MX", "GOOGL": "GOOGL.MX",
    "CUERVO": "CUERVO.MX", "FEMSA": "FEMSAUBD.MX", "TSLA": "TSLA",
    "PYPL": "PYPL", "SBUX": "SBUX", "DIS": "DIS", "AMZN": "AMZN.MX",
    "NKE": "NKE", "GFNORTE": "GFNORTEO.MX", "WALMEX": "WALMEX.MX",
    "CEMEX": "CEMEXCPO.MX", "CLF": "CLF.MX", "CVX": "CVX.MX",
    "BIMBO": "BIMBOA.MX", "GAP": "GAPB.MX", "NFLX": "NFLX.MX",
    "ASUR": "ASURB.MX", "MA": "MA.MX", "KOF": "KOFUBL.MX", "V": "V.MX",
    "MU": "MU", "AVGO": "AVGO", "NVDA": "NVDA", "AMD": "AMD",
    "MSFT": "MSFT", "AAPL": "AAPL", "MRNA": "MRNA",
    "EWZ": "EWZ",  # ETF de Brasil, sin apalancamiento
}

In [4]:
def metricas(ticker):
    precios = yf.Ticker(ticker).history(period="1y")["Close"].dropna()
    if len(precios) < 130:
        return None

    ret = precios.pct_change().dropna()
    delta = precios.diff()
    ganancia = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    perdida = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    rsi = 100 - 100 / (1 + ganancia / perdida)

    ma20_serie = precios.rolling(20).mean()

    return {
        "Precio": precios.iloc[-1],
        "Mom_3m": precios.iloc[-1] / precios.iloc[-63] - 1,
        "Mom_6m": precios.iloc[-1] / precios.iloc[-126] - 1,
        "Vol": ret.iloc[-60:].std() * np.sqrt(252),
        "MA20": ma20_serie.iloc[-1],
        "MA50": precios.rolling(50).mean().iloc[-1],
        "RSI": rsi.iloc[-1],
        # Cruce hacia arriba de la MA20: hoy por encima y ayer por debajo o igual
        "Cruce_MA20": int(precios.iloc[-1] > ma20_serie.iloc[-1]
                          and precios.iloc[-2] <= ma20_serie.iloc[-2]),
    }

In [5]:
def reparte(indices, total):
    return {t: total / len(indices) for t in indices} if len(indices) else {}

In [6]:
filas = {}
for simbolo, ticker in universo.items():
    try:
        m = metricas(ticker)
        if m:
            filas[simbolo] = m
    except Exception:
        pass

In [7]:
df = pd.DataFrame(filas).T.astype(float)
df["Score"] = df["Mom_6m"] / df["Vol"]

In [8]:
df = df[df["Vol"] < 1.5]

In [9]:
nucleo = df[(df["Precio"] > df["MA50"]) & (df["Vol"] < VOL_MAX_NUCLEO) & (df["RSI"] < RSI_MAX_NUCLEO)]
nucleo = nucleo.sort_values("Score", ascending=False).head(3)

In [10]:
sat = df[(df["Precio"] > df["MA50"]) & (df["Vol"] < VOL_MAX_SATELITE)]
if RSI_MAX_SATELITE is not None:
    sat = sat[sat["RSI"] < RSI_MAX_SATELITE]
satelite = sat.drop(nucleo.index, errors="ignore")
satelite = satelite.sort_values("Mom_6m", ascending=False).head(3)

In [11]:
reb = df[(df["RSI"] < 40) & (df["Precio"] < df["MA20"]) & (df["Mom_6m"] > 0)]
rebote = reb.drop(nucleo.index.union(satelite.index), errors="ignore")
rebote = rebote.sort_values("RSI").head(3)

In [12]:
pesos = {}
pesos.update(reparte(nucleo.index, 1 - PESO_SATELITE_REBOTE))
pesos.update(reparte(satelite.index, PESO_SATELITE_REBOTE * 0.6))
pesos.update(reparte(rebote.index, PESO_SATELITE_REBOTE * 0.4))

In [13]:
total = sum(pesos.values())
pesos = {k: v / total for k, v in pesos.items()}

In [14]:
assert len(pesos) >= 5, "Necesitas al menos 5 instrumentos"
assert max(pesos.values()) <= 0.50, "Ninguna posición puede pasar 50%"
assert abs(sum(pesos.values()) - 1) < 1e-9, "Los pesos deben sumar 100%"

In [15]:
cols = ["Mom_6m", "Vol", "RSI", "Score"]
print("NÚCLEO\n", nucleo[cols].round(2))
print("\nSATÉLITE\n", satelite[cols].round(2))
print("\nREBOTE\n", rebote[cols].round(2))
print("\nPesos:", {k: f"{v:.1%}" for k, v in pesos.items()})

NÚCLEO
       Mom_6m   Vol    RSI  Score
MSFT    0.32  0.26  63.35   1.24
KOF     0.26  0.23  67.26   1.14
CVX     0.28  0.27  51.19   1.05

SATÉLITE
      Mom_6m   Vol    RSI  Score
V      0.44  0.21  80.59   2.09
MA     0.43  0.23  77.91   1.90
CRM    0.43  0.60  50.83   0.72

REBOTE
      Mom_6m   Vol    RSI  Score
MU     0.24  0.57  35.67   0.41
AMD    0.18  0.60  38.36   0.30

Pesos: {'MSFT': '11.7%', 'KOF': '11.7%', 'CVX': '11.7%', 'V': '13.0%', 'MA': '13.0%', 'CRM': '13.0%', 'MU': '13.0%', 'AMD': '13.0%'}


In [16]:
ventas = [t for t in MIS_REBOTE if t in df.index and df.loc[t, "Cruce_MA20"] == 1]
print("\nSEÑALES DE VENTA (rebotes que cruzaron hacia arriba MA20):", ventas if ventas else "ninguna")


SEÑALES DE VENTA (rebotes que cruzaron hacia arriba MA20): ninguna


In [17]:
candidatas = df[(df["Precio"] > df["MA50"]) & (df["RSI"] < RSI_MAX_NUCLEO)]
candidatas = candidatas.sort_values("Score", ascending=False)
if not candidatas.empty:
    mejor = candidatas.index[0]
    print(f"\n>>> MEJOR SCORE: {mejor} (Score {candidatas['Score'].iloc[0]:.2f}, "
          f"Mom 6m {candidatas['Mom_6m'].iloc[0]:.1%}, Vol {candidatas['Vol'].iloc[0]:.1%})")
else:
    print("\n>>> Ninguna emisora cumple tendencia alcista y RSI bajo hoy.")


>>> MEJOR SCORE: MSFT (Score 1.24, Mom 6m 31.9%, Vol 25.7%)
